Analyse Building and Fitting a model in GELATO
----------------------------------------------

Copy from ExampleFitInNb_recoverfitfailure.ipynb

- author : Sylvie Dagoret-Campagne
- creation date : 2024-07-05
- last update : 2024-08-27 (add version 4 for spectra)
- last update : 2024-08-30 (add version 3_debug for spectra)

- Kernel at CCIN2P3 : ``conda_desc_py310_pcigale``
- Kernel on my laptop : ``pcigale``

In [ ]:
# Import packages
import gelato
import numpy as np
%matplotlib inline
import matplotlib as mpl
mpl.rcParams['font.size'] = 25
from matplotlib import pyplot # For plotting
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# For loading in data
from astropy.io import fits
from astropy.table import Table 
import os,re
import pandas as pd

In [ ]:
props = dict(boxstyle='round', facecolor='wheat', alpha=1.0)

In [ ]:
from astropy.modeling import models, fitting
from astropy import modeling
# define a model for a line
g_init = models.Gaussian1D(amplitude=1, mean=0, stddev=1)
# initialize a linear fitter
fit_g = fitting.LevMarLSQFitter()

In [ ]:
from fors2pcigale.fors2starlightio import Fors2DataAcess

In [ ]:
#from gelato.Plotting import  Plot, PlotFig,subplotplot
#from gelato.Plotting import subplotplot
from gelato.Plotting import logbarrier
from scipy.optimize import minimize

#import gelato.ConstructParams as CP

import gelato.Utility as U
import gelato.Plotting as P
import gelato.ConstructParams as CP

# GELATO
import gelato.Utility as U
import gelato.CustomModels as CM
import gelato.SpectrumClass as SC

from gelato.Constants import C

In [ ]:
from gelato.ModelComparison import Chi2

In [ ]:
# Packages
import copy
import numpy as np
from itertools import combinations
from scipy.optimize import least_squares

# gelato supporting files
import gelato.Utility as U
import gelato.BuildModel as BM
import gelato.CustomModels as CM
import gelato.ModelComparison as MC
import gelato.AdditionalComponents as AC

import gelato.FittingModel as FM

In [ ]:
from libExampleFitInNb import *

In [ ]:
import shutil
import json

## Fors2 Interface

In [ ]:
fors2 = Fors2DataAcess()

## Gelato Parameters

In [ ]:
version = "v3"
#mode = ""
mode = "_debug"

In [ ]:
# Path to the parameters file
#path_params = './ExampleParametersFitInNb.json'
path_params = f"./ExampleParametersFitInNb_{version}{mode}.json"

# Create Parameters dictionary
params_gel = gelato.ConstructParams.construct(path_params)

# Set to not multiprocessing
params_gel['NProcess'] = 1

In [ ]:
params_gel['EmissionGroups']

In [ ]:
for group in params_gel['EmissionGroups']:
    info_group = "Group : "+ group["Name"]
    print(info_group)
    all_species = group['Species']
    for the_species in all_species:
        #info_species = "\t  Species : " + the_species['Name'] + " FlagGroup :: " +  the_species['FlagGroups'][0] + "Nlines = " + str(len( the_species['Lines'] )) 
        info_species = "\t  Species : " + the_species['Name']  + " , Nlines = " + str(len( the_species['Lines'] )) 
        print(info_species)

## Table with Spectra name and Redshifts

In [ ]:
#df = pd.read_csv("object_filelist_v0.csv",index_col=0)
filename_object_file_list = f"object_filelist_{version}.csv"
df_objectslist = pd.read_csv(filename_object_file_list,index_col=0)

## Input files before the fit, sorting and index

In [ ]:
path = f"./spec_forgelato/{version}"

In [ ]:
list_all_files = os.listdir(path)

In [ ]:
idx_selected_files = []
list_selected_files = []
for file in list_all_files:
    res = re.findall("^specgelato_SPEC.*[.]fits$",file)
    if len(res):
        list_selected_files.append(file)
        num = int(re.findall("specgelato_SPEC(.*)[.]fits$",file)[0])   
        idx_selected_files.append(num)

In [ ]:
idx_selected_files = np.array(idx_selected_files)
list_selected_files = np.array(list_selected_files)
idx_sorted_files = np.argsort(idx_selected_files)
list_sorted_files = list_selected_files[idx_sorted_files]

In [ ]:
NSPEC = len(list_sorted_files)

## Choose One file

In [ ]:
index=0 # the original spectrum baseline is shifted

#index = 1 
#index = 8

#index = 32
#index = 27
shortfilename = list_sorted_files[index]
fullfilename = os.path.join(path,shortfilename) 
path_spec = fullfilename
tag_spec = re.findall(".*_(SPEC.*).fits$", shortfilename)

In [ ]:
shortfilename.split('.')[0]

In [ ]:
tag_spec

### Define output filenames

#### Fit results

In [ ]:
output_filename = shortfilename.split('.')[0] + "-results.fits"

In [ ]:
output_filename

#### pulls and emission-line results

In [ ]:
output_filename_pulls = shortfilename.split('.')[0] + "-pulls-results.csv"
output_filename_emissionlines = shortfilename.split('.')[0] + "-emissionlines-results.csv"

#### Find the redshift

In [ ]:
if len(tag_spec)>0:
    tag_spec = tag_spec[0]
    all_inputspecfilenames = df_objectslist.Path.values
    for idx_tag,filen in enumerate(all_inputspecfilenames):
        if tag_spec in filen:
                break

    df_row = df_objectslist.iloc[idx_tag]
    redshift = df_row["z"]

In [ ]:
spec_name_sel = tag_spec

In [ ]:
spec_name_sel

In [ ]:
# acess to the image array
img = fors2.get_specimg(spec_name_sel)

# get the image filename and path
spec_sec_fileimg = fors2.get_specimgfile(spec_name_sel)

In [ ]:
fig = plt.figure(constrained_layout=True,figsize=(12,6))
plt.imshow(img)
ax = plt.gca()
# Hide X and Y axes label marks
ax.xaxis.set_tick_params(labelbottom=False)
ax.yaxis.set_tick_params(labelleft=False)
# Hide X and Y axes tick marks
ax.set_xticks([])
ax.set_yticks([])
plt.show()

In [ ]:
print(df_row,redshift)

In [ ]:
title = f"{index}) {output_filename}, z={redshift:.3f}" 

#### Get the spectrum

In [ ]:
spectrum = Table.read(path_spec)

# Start with inverse variance
ivar = spectrum['ivar']
good = ivar > 0 # GELATO only looks at points with nonzero weights

# Finally, let's load in the data
wavl = 10**spectrum['loglam'][good]
flux = spectrum['flux'][good]
ivar = ivar[good]
args = (wavl,flux,ivar) # These will be useful later

In [ ]:
spectrum[:5]

Let's go ahead and plot our spectrum to get an idea of what we're dealing with.

In [ ]:
# Create figure
fig, ax = pyplot.subplots(figsize=(15,7))

# Plot Spectrum
sig = 3/np.sqrt(ivar) # 3 Sigma boundary
ax.fill_between(wavl,flux-sig,flux+sig,color='gray')
ax.step(wavl,flux,where='mid',c='k',lw=0.5)

# Axis limits
ax.set(xlim=[wavl.min(),wavl.max()],ylim=[0,flux.max()])

# Axis labels
ax.set(xlabel=r'Obs. Wavelength [\AA]',ylabel=r'$F_\lambda$')
ax.set_title(title)
# Show figure
pyplot.show()

The main gelato function takes three inputs.
* The path to the parameters file or the parameters dictionary.
* The path to the spectrum.
* The redshift of the spectrum.

We already have the last two, and we need to take a little precaution with the first.
The main gelato function will only return the final model if the code is being run without multiprocessing (as the return statement can break Python multiprocessing). So we can either change the Parameters JSON file, or edit the parameters dictionary. 

## Output for results

In [ ]:
output_path = params_gel['OutFolder']

In [ ]:
if not os.path.isdir(output_path):
    os.mkdir(output_path)

In [ ]:
print(f"output_path defined in json file : {output_path}")

In [ ]:
output_path_fullfilename = os.path.join(output_path,output_filename)
output_path_fullfilename_pulls = os.path.join(output_path,output_filename_pulls)
output_path_fullfilename_emissionlines = os.path.join(output_path,output_filename_emissionlines)

## Run Gelato Fit

We are now ready to run GELATO. Note, before you do this, ensure the results directory exists, either by running the Example from the README file or creating it. It will return the final callable model, however it won't be used in this notebook. 

In [ ]:
model = gelato.gelato(params_gel,path_spec,redshift)

## Results of Gelato Fit

The results have been saved to the "Results/" Directory. Let's go ahead and load them in. We will print all extensions on the folder.

### 1) results

In [ ]:
# Load in results
results = fits.open(output_path_fullfilename)

# Print FITS extensions
results.info()

We have two FITS extensions, SUMMARY and PARAMS. They are described in more detail in the README File but let's play around with them directly. Let's go ahead and take a look inside the SUMMARY extension. As we can see, it is a binary FITS Table.

### Summary of fitted model

In [ ]:
summary = Table(results['SUMMARY'].data)
summary

In this table, we have the original spectrum along with the various model components, we can go ahead and plot them.

In [ ]:
df = summary.to_pandas()

In [ ]:
# Create figure
fig, ax = pyplot.subplots(figsize=(15,3))

# Plot Spectrum
ax.step(10**summary['loglam'],summary['LINE'],where='mid',c='y',label='Emission Lines')
ax.legend()

# Axis limits
ax.set(xlim=[wavl.min(),wavl.max()],ylim=[0,flux.max()])
# Axis labels
ax.set(xlabel=r'Obs. Wavelength [\AA]',ylabel=r'$F_\lambda$')
ax.set_title(title)
ax.grid()
# Show figure
pyplot.show()

In [ ]:
# Create figure
fig, ax = pyplot.subplots(figsize=(15,7))

# Plot Spectrum
ax.fill_between(wavl,flux-sig,flux+sig,color='gray')
ax.step(wavl,flux,where='mid',c='k',lw=0.5,label='Data')
ax.step(10**summary['loglam'],summary['MODEL'],where='mid',c='r',label='Total Model')
ax.step(10**summary['loglam'],summary['SSP'],where='mid',c='g',label='SSP Cont.')
#ax.step(10**summary['loglam'],summary['PL'],where='mid',c='b',label='Power-Law Cont.')
ax.step(10**summary['loglam'],summary['LINE'],where='mid',c='y',label='Emission Lines')
ax.legend()

# Axis limits
ax.set(xlim=[wavl.min(),wavl.max()],ylim=[0,flux.max()])

# Axis labels
ax.set(xlabel=r'Obs. Wavelength [\AA]',ylabel=r'$F_\lambda$')
ax.set_title(title)
# Show figure
pyplot.show()

Looks great! You can see an example of the GELATO generated plots in the results folder, but this will let you incorporate GELATO fits easily into your own work. Let's go ahead and take a look at the PARAMS extension. This is a much larger table! It's made up of the parameters from each bootstrap iteration. 

### A very simple view of fit results 

In [ ]:
MySimplePlotSpectrumWithFittedModel(output_path_fullfilename,redshift,title)

### 2) Access to the fitted spectrum object directly

In [ ]:
spectrum = SC.Spectrum(output_path_fullfilename,redshift,params_gel)

In [ ]:
spectrum.p

In [ ]:
spectrum.p["EmissionGroups"]

In [ ]:
len(spectrum.p["EmissionGroups"])

In [ ]:
spectrum.regions

In [ ]:
region_inv = np.invert(spectrum.emission_region)
region_inv

In [ ]:
fig, ax1 = plt.subplots(1,1,figsize=(10,4))
ax1.plot(spectrum.wav,spectrum.flux,'b-')
ax2= ax1.twinx()
ax2.plot(spectrum.wav,region_inv,"g:")
ax1.plot(spectrum.wav[region_inv],spectrum.flux[region_inv],'r:')
title = f"Selected regions to fit continuum for {spec_name_sel}"
ax1.set_title(title)
ax1.set(xlabel=r'Obs. Wavelength [\AA]',ylabel=r'$F_\lambda$')


### Input redshift

In [ ]:
spectrum.z

In [ ]:
Z_init = spectrum.z*C
Z_init

## Debug Model

In [ ]:
from gelato.FittingModel import *

### Fit the continuum to adjust the redshift

In [ ]:
cont,cont_x = FM.FitContinuum(spectrum)

In [ ]:
print(cont.get_names())

In [ ]:
type(cont.models[0])

In [ ]:
len(cont.models)

In [ ]:
models = cont.models

##### adjusted redshift

In [ ]:
Z_fit = cont.models[0].redshift
Z_fit

In [ ]:
dz = (Z_fit-Z_init)/C
dz

In [ ]:
len(cont.models[0].ssps)

In [ ]:
len(cont_x)

##### fitted coefficients

In [ ]:
cont_x

In [ ]:
model0 = cont.models[0]
print("* continuous model 0 name :: \n ",model0.get_names())

if len(cont.models)>1:
    model1 = cont.models[1]
    print(model1.get_names())
    print("* continuous model 1 name :: \n",model1.get_names())
    

In [ ]:
args = spectrum.wav,spectrum.flux,spectrum.isig



#if 'PowerLaw_Index' in model.get_names():
if len(models)>1:
    continuum = CM.CompoundModel(models[0:2]).evaluate(cont_x,*args)
    residuals = -CM.CompoundModel(models[0:2]).residual(cont_x,*args) 
    chi2 = Chi2(CM.CompoundModel(models[0:2]),cont_x,args)
    nparams = len(cont_x)
else: 
    continuum = CM.CompoundModel(models[0:1]).evaluate(cont_x,*args)
    residuals = -CM.CompoundModel(models[0:1]).residual(cont_x,*args) 
    chi2 = Chi2(CM.CompoundModel(models[0:1]),cont_x,args)
    nparams = len(cont_x)


In [ ]:
spectrum.regions

### Result of continuum fit

In [ ]:
fig, (ax1,ax3) = plt.subplots(2,1,figsize=(10,8))

Ndata = 0
for i,region in enumerate(spectrum.regions):
    # Get Spectrum
    good    = np.logical_and(spectrum.wav < region[1],spectrum.wav > region[0])
    wav     = spectrum.wav[good]
    flux    = spectrum.flux[good]
    isig    = spectrum.isig[good]
    args    = wav,flux,isig
    Ndata+= len(wav)
    
    #plt.step(wav,continuum[good],where='mid')
    ax1.plot(wav,flux,"-",c="grey")
    ax1.step(wav,continuum[good],where='mid')
    ax1.axvline(region[0],ls="-.",color="r")
    ax1.axvline(region[1],ls="-.",color="r")
    
ax2= ax1.twinx()
ax2.plot(spectrum.wav,region_inv,"g:")

title1 = f"Regions outside region used to fit continuum for {spec_name_sel}"
ax1.set_title(title1)
ax1.grid()
ax1.set(xlabel=r'Obs. Wavelength [\AA]',ylabel=r'$F_\lambda$')




chi2red = chi2/(Ndata-nparams)
chi2label = f"Chi2 = {chi2:.2f}, Chi2_red = {chi2red:.2f}"



#plt.step(wav,continuum[good],where='mid')
ax3.plot(spectrum.wav ,residuals,"-",c="grey")
ax3.grid()
ax3.set(xlabel=r'Obs. Wavelength [\AA]',ylabel=r'$F_\lambda$')
title3 = f"residuals after continuum fit for {spec_name_sel}"
ax3.set_title(title3)

ax3.text(0.05, 0.95, chi2label, transform=ax3.transAxes, fontsize=12,verticalalignment='top',bbox=props)

plt.tight_layout()

In [ ]:
fig, (ax1,ax3) = plt.subplots(2,1,figsize=(10,8))
wav     = spectrum.wav[region_inv]
flux    = spectrum.flux[region_inv]
isig    = spectrum.isig[region_inv]
args    = wav,flux,isig
    #plt.step(wav,continuum[good],where='mid')
ax1.plot(wav,flux,"-",color="grey")
ax1.step(wav,continuum[region_inv],"b",where='mid')
ax1.text(0.05, 0.95, chi2label, transform=ax1.transAxes, fontsize=12,verticalalignment='top',bbox=props)
ax2= ax1.twinx()
ax2.plot(spectrum.wav,region_inv,"g:")

title1 = f"Continuum portions used to fit continuum for {spec_name_sel}"
ax1.set_title(title1)
ax1.set(xlabel=r'Obs. Wavelength [\AA]',ylabel=r'$F_\lambda$')

ax3.plot(spectrum.wav ,residuals,"-",c="grey")
ax3.grid()
ax3.set(xlabel=r'Obs. Wavelength [\AA]',ylabel=r'$F_\lambda$')
title3 = f"residuals to fit continuum for {spec_name_sel}"
ax3.set_title(title3)

plt.tight_layout()


## Fit Emission Lines

In [ ]:
# Find number of flags
flags = 0
for group in spectrum.p['EmissionGroups']:
    for species in group['Species']:    
        flagbits = bin(species['Flag'])[2:]
        flags += len(flagbits.replace('0',''))
        print("g=",group["Name"],"\t s=",species["Name"],"\t species['Flag'] = ",species['Flag'] ,"\t flagbits=",flagbits,"\t flags=",flags)

In [ ]:
bin(3)[2:]

In [ ]:
for i in range(flags):    
    # Add new component
    EmissionGroups = AddComplexity(spectrum.p['EmissionGroups'],i)
    print(f"* AddComplexity :: flag={i} , EmissionGroups = ",EmissionGroups)


### Build the emission model

In [ ]:
# Build emission line model
emis,emis_x = BM.BuildEmission(spectrum)

In [ ]:
type(emis)

In [ ]:
print("* Parameters for emission ::\n",emis.get_names())

In [ ]:
emis_x,len(emis_x)

### Fit the lines

In [ ]:
#model,model_fit = FM.FitComponents(spectrum,cont,cont_x,emis,emis_x)

In [ ]:
# Fit region
args = (spectrum.wav,spectrum.flux,spectrum.isig)

# Base Model
cont.models[0].bounds = tuple((cont_x[i]*(1-1e-5),cont_x[i]*(1+1e-5)) for i in range(cont.models[0].nparams))
constraints = BM.TieParams(spectrum,cont.get_names()+emis.get_names())
base_model,x0 = BM.BuildModel(spectrum,cont,cont_x,emis,emis_x,constraints)

# Initial fit
x0 = base_model.constrain(x0) # Limit to true parameters

fit_result = FitModel(base_model,x0,args,jac=base_model.jacobian)
base_fit_msg = fit_result.message
base_fit = fit_result.x
#base_fit = FitModel(base_model,x0,args,jac=base_model.jacobian).x

if spectrum.p["Verbose"]:
    print(">>>> FitComponents :: FitModel ==> message =",base_fit_msg, " res = ",base_fit)

In [ ]:
x0

In [ ]:
cont_x

In [ ]:
base_model.get_names()

In [ ]:
type(model) 

In [ ]:
fittedspectrum = CM.CompoundModel(model).evaluate(cont_x,*args)
residuals = -CM.CompoundModel(model).residual(cont_x,*args) 
chi2 = Chi2(CM.CompoundModel(model),cont_x,args)
    


In [ ]:
len(model_fit)

In [ ]:
print("* Parameters for FitComponents ::\n",model.get_names())

In [ ]:
len(model.get_names())

In [ ]:
len(model_fit) + len(cont_x)

In [ ]:
model_fit